# Notebook 2: Damage Detection Model (Local, YOLOv8s)

**Goal:** Train YOLOv8s on the Automobile Damage dataset (14 classes).

**Config:** yolov8s + 100 epochs + patience=20 + batch=16
**Expected on RTX 5070:** ~1.5-2 hours total. First epoch is slowest (image caching).

**Target:** mAP@0.5 ≥ 0.55. Realistically 0.55-0.75 depending on how hard the class imbalance hits.


## 1. Load paths

In [1]:
import os, json, torch

with open('dataset_paths.json') as f:
    paths = json.load(f)

print("Detection yaml:", paths['detection_yaml'])
print("CUDA available:", torch.cuda.is_available())
print("Device:        ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Detection yaml: d:\VIT\Sem-5\AI\Project\New\data\automobile_damage\data.yaml
CUDA available: True
Device:         NVIDIA GeForce RTX 5070 Laptop GPU


**Sanity check:** CUDA must be True and device must be your RTX 5070. If not, training will fall back to CPU and take days.


## 2. Train

In [2]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # downloads ~22 MB the first time

results = model.train(
    data=paths['detection_yaml'],
    epochs=70,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,                # force GPU
    project='runs',
    name='damage_detection',
    exist_ok=True,
    save=True,
    plots=True,
    workers=4,               # data loading threads
)

Ultralytics 8.4.96  Python-3.11.9 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\VIT\Sem-5\AI\Project\New\data\automobile_damage\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=damage_detection, n

### While training runs

- Loss values (`box_loss`, `cls_loss`, `dfl_loss`) should decrease steadily
- After each epoch, validation metrics print: **mAP50**, **mAP50-95**
- Rough trajectory: epoch 10 → mAP50 ~0.30, epoch 30 → ~0.45, epoch 60+ → 0.55-0.70
- With `patience=20`, training auto-stops if mAP hasn't improved for 20 consecutive epochs
- The **best** weights are saved to `runs/damage_detection/weights/best.pt` automatically

### If VS Code disconnects mid-training

Training runs in the Python kernel — as long as your laptop stays on and doesn't sleep, it keeps going even if you close VS Code (the kernel process survives).

**But:** if the whole laptop sleeps or restarts, training stops. Ultralytics saves checkpoints after every epoch, so resume from `last.pt` with `resume=True`.


## 3. Evaluate best model on validation set

In [3]:
metrics = model.val()

print(f"\nOverall mAP@0.5     : {metrics.box.map50:.3f}")
print(f"Overall mAP@0.5:0.95: {metrics.box.map:.3f}")
print(f"Precision          : {metrics.box.mp:.3f}")
print(f"Recall             : {metrics.box.mr:.3f}")

Ultralytics 8.4.96  Python-3.11.9 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
Model summary (fused): 73 layers, 11,131,002 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1171.5550.2 MB/s, size: 95.9 KB)
val: Scanning D:\VIT\Sem-5\AI\Project\New\data\automobile_damage\valid\labels.cache... 1043 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1043/1043  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 66/66 8.8it/s 7.5s0.1s
                   all       1043       1610       0.88       0.76      0.828      0.626
Front-windscreen-damage         49         50       0.86      0.735      0.761      0.628
      Headlight-damage         89         90       0.78      0.747      0.802      0.563
Rear-windscreen-Damage         77         77          1      0.845      0.895      0.722
   Runningboard-Damage         43         50      0.798       0.74     

## 4. Look at the training curves

In [5]:
from IPython.display import Image, display
import os

run_dir = 'runs/damage_detection'
for fname in ['results.png', 'confusion_matrix.png', 'F1_curve.png', 'PR_curve.png']:
    fpath = os.path.join(run_dir, fname)
    if os.path.exists(fpath):
        print(fname)
        display(Image(fpath))

## 5. Test on unseen validation images

In [ ]:
import glob, cv2
import matplotlib.pyplot as plt

test_imgs = sorted(glob.glob(f"{paths['detection_dir']}/valid/images/*.jpg"))[:6]
results = model.predict(test_imgs, conf=0.25, save=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, r in zip(axes.flat, results):
    annotated = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(annotated); ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Copy best weights to models/

In [7]:
import shutil, os

os.makedirs('models', exist_ok=True)

# Actual path (Ultralytics adds detect/ subfolder)
src = 'runs/detect/runs/damage_detection/weights/best.pt'

if not os.path.exists(src):
    # Fall back to searching
    for root, dirs, files in os.walk('runs'):
        for f in files:
            if f == 'best.pt':
                src = os.path.join(root, f)
                print("Found best.pt at:", src)
                break

shutil.copy(src, 'models/detection_best.pt')
print(f"Saved: models/detection_best.pt ({os.path.getsize('models/detection_best.pt')/1e6:.1f} MB)")

Saved: models/detection_best.pt (22.5 MB)


## Done

- ✅ YOLOv8s trained
- ✅ `models/detection_best.pt` saved

**Next:** Notebook 3 — severity classifier (CarDD data, ResNet-18, ~10-15 min training).